# Projekt 1 â€” Å›wiat klockÃ³w

Raport dla progu 8 punktÃ³w. Eksperymenty uÅ¼ywajÄ… ziarna `42` i limitu `1 000 000` rozwiniÄ™tych wÄ™zÅ‚Ã³w. Przekroczenie limitu jest raportowane, a nie pomijane.

In [1]:
import json
from pathlib import Path
from statistics import mean
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
RESULTS_PATH = ROOT / 'output' / 'results.json'
SEED = 42
MAX_EXPANDED = 1_000_000

assert RESULTS_PATH.exists(), f'Brak wymaganego pliku {RESULTS_PATH}'
records = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
assert len(records) == 135, f'Oczekiwano 135 rekordÃ³w, otrzymano {len(records)}'
invalid = [row['case_id'] for row in records if row['status'] == 'invalid_solution']
assert not invalid, f'Niepoprawne rozwiÄ…zania: {invalid}'
print(f'Wczytano {len(records)} rekordÃ³w z {RESULTS_PATH}')

Wczytano 135 rekordÃ³w z C:\Users\piotr\Projects\Blocks-World\output\results.json


In [2]:
def format_state(state):
    return ' '.join(map(str, state))

def format_moves(moves):
    return ' â†’ '.join(
        f'{block}â†’{"stÃ³Å‚" if destination == 0 else destination}'
        for block, destination in moves
    ) or 'â€”'

def format_cell(column, value):
    if column in {'start', 'goal'}:
        return format_state(value)
    if column == 'moves':
        return format_moves(value)
    if isinstance(value, float):
        return f'{value:.4f}'
    return 'â€”' if value is None else str(value)

def table_markdown(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join('---' for _ in columns) + ' |'
    body = [
        '| ' + ' | '.join(format_cell(column, row.get(column)) for column in columns) + ' |'
        for row in rows
    ]
    return '\n'.join([header, separator, *body])

def show_table(rows, columns):
    display(Markdown(table_markdown(rows, columns) if rows else '_Brak wynikÃ³w dla tej sekcji._'))

def show_details(label, rows, columns):
    if rows:
        display(Markdown(
            f'<details><summary>{label}</summary>\n\n'
            f'{table_markdown(rows, columns)}\n\n</details>'
        ))

def verified(rows):
    return [row for row in rows if row['verification'] == 'passed']

## Reprezentacja i kontrola poprawnoÅ›ci

Stan jest niemutowalnÄ… krotkÄ…. Element na pozycji `i` okreÅ›la bezpoÅ›rednie podÅ‚oÅ¼e klocka `i+1`, a `0` oznacza stÃ³Å‚. Walidator odrzuca nieistniejÄ…ce podÅ‚oÅ¼a, samopodparcie, dwa klocki na jednym podÅ‚oÅ¼u i cykle. KaÅ¼da sekwencja zwrÃ³cona przez solver jest ponownie wykonywana przez niezaleÅ¼ny weryfikator, ktÃ³ry wskazuje pierwszy niepoprawny ruch.

## Wariant podstawowy â€” DFS

DFS z nawrotami pomija stany juÅ¼ odwiedzone i koÅ„czy pracÄ™ po osiÄ…gniÄ™ciu limitu. Ruchy sÄ… rozpatrywane w deterministycznej kolejnoÅ›ci generatora: ruchy na stÃ³Å‚ wedÅ‚ug numeru klocka, a nastÄ™pnie ruchy na klocek leksykograficznie po `(klocek, cel)`. Jest to kolejnoÅ›Ä‡ z implementacji DFS kolegi; DFS zwraca pierwsze znalezione rozwiÄ…zanie i nie gwarantuje jego minimalnej dÅ‚ugoÅ›ci.

In [3]:
dfs_basic = [
    row for row in records
    if row['algorithm'] == 'dfs' and row['variant'] == 'basic'
]
dfs_summary = []
for n in range(3, 16):
    rows = [row for row in dfs_basic if row['n'] == n]
    dfs_summary.append({
        'n': n,
        'przypadki': len(rows),
        'rozwiÄ…zane': sum(row['status'] == 'solved' for row in rows),
        'limit': sum(row['status'] == 'node_limit' for row in rows),
        'Å›rednio wÄ™zÅ‚Ã³w': round(mean(row['expanded_nodes'] for row in rows), 1),
        'Å›rednio sekund': round(mean(row['elapsed_seconds'] for row in rows), 3),
    })
show_table(dfs_summary, list(dfs_summary[0]))

examples = [
    {
        'case_id': row['case_id'],
        'przypadek': f"{format_state(row['start'])}, {format_state(row['goal'])}",
        'ruchy': format_moves(row['moves']),
    }
    for row in verified(dfs_basic)[:5]
]
display(Markdown('**PrzykÅ‚adowe rozwiÄ…zania**'))
show_table(examples, ['case_id', 'przypadek', 'ruchy'])
show_details(
    'PokaÅ¼ peÅ‚ne wyniki DFS (65 przypadkÃ³w)',
    dfs_basic,
    ['case_id', 'n', 'start', 'goal', 'status', 'move_count', 'expanded_nodes', 'elapsed_seconds', 'verification'],
)

| n | przypadki | rozwiÄ…zane | limit | Å›rednio wÄ™zÅ‚Ã³w | Å›rednio sekund |
| --- | --- | --- | --- | --- | --- |
| 3 | 5 | 5 | 0 | 4.2000 | 0.0000 |
| 4 | 5 | 5 | 0 | 10.6000 | 0.0000 |
| 5 | 5 | 5 | 0 | 271.8000 | 0.0010 |
| 6 | 5 | 5 | 0 | 1354.8000 | 0.0060 |
| 7 | 5 | 5 | 0 | 9314.4000 | 0.0450 |
| 8 | 5 | 5 | 0 | 283000.2000 | 4.0200 |
| 9 | 5 | 1 | 4 | 821061 | 16.9590 |
| 10 | 5 | 0 | 5 | 1000000 | 27.3460 |
| 11 | 5 | 0 | 5 | 1000000 | 33.3870 |
| 12 | 5 | 0 | 5 | 1000000 | 38.6020 |
| 13 | 5 | 0 | 5 | 1000000 | 50.3920 |
| 14 | 5 | 0 | 5 | 1000000 | 62.6270 |
| 15 | 5 | 0 | 5 | 1000000 | 75.7890 |

**PrzykÅ‚adowe rozwiÄ…zania**

| case_id | przypadek | ruchy |
| --- | --- | --- |
| problem_n03_01 | 2 0 0, 3 0 0 | 1â†’stÃ³Å‚ â†’ 1â†’3 |
| problem_n03_02 | 0 0 0, 0 0 2 | 3â†’2 |
| problem_n03_03 | 0 3 0, 0 0 0 | 2â†’stÃ³Å‚ |
| problem_n03_04 | 3 0 0, 0 3 0 | 1â†’stÃ³Å‚ â†’ 2â†’3 |
| problem_n03_05 | 0 1 0, 0 0 0 | 2â†’stÃ³Å‚ |

<details><summary>PokaÅ¼ peÅ‚ne wyniki DFS (65 przypadkÃ³w)</summary>

| case_id | n | start | goal | status | move_count | expanded_nodes | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| problem_n03_01 | 3 | 2 0 0 | 3 0 0 | solved | 2 | 2 | 0.0000 | passed |
| problem_n03_02 | 3 | 0 0 0 | 0 0 2 | solved | 1 | 11 | 0.0001 | passed |
| problem_n03_03 | 3 | 0 3 0 | 0 0 0 | solved | 1 | 1 | 0.0000 | passed |
| problem_n03_04 | 3 | 3 0 0 | 0 3 0 | solved | 2 | 6 | 0.0001 | passed |
| problem_n03_05 | 3 | 0 1 0 | 0 0 0 | solved | 1 | 1 | 0.0001 | passed |
| problem_n04_01 | 4 | 0 1 0 3 | 0 1 2 0 | solved | 8 | 12 | 0.0001 | passed |
| problem_n04_02 | 4 | 0 0 4 0 | 0 0 0 0 | solved | 1 | 1 | 0.0000 | passed |
| problem_n04_03 | 4 | 4 0 0 3 | 0 1 0 0 | solved | 7 | 11 | 0.0001 | passed |
| problem_n04_04 | 4 | 0 3 4 1 | 2 0 0 1 | solved | 5 | 26 | 0.0004 | passed |
| problem_n04_05 | 4 | 0 1 2 0 | 2 0 0 0 | solved | 3 | 3 | 0.0000 | passed |
| problem_n05_01 | 5 | 4 0 5 0 0 | 5 4 0 0 0 | solved | 90 | 380 | 0.0012 | passed |
| problem_n05_02 | 5 | 4 0 0 3 2 | 0 4 0 1 0 | solved | 65 | 373 | 0.0015 | passed |
| problem_n05_03 | 5 | 2 0 0 0 0 | 5 4 0 3 0 | solved | 64 | 253 | 0.0008 | passed |
| problem_n05_04 | 5 | 0 1 0 0 0 | 0 0 5 2 0 | solved | 77 | 223 | 0.0006 | passed |
| problem_n05_05 | 5 | 5 0 0 0 4 | 5 1 0 0 0 | solved | 52 | 130 | 0.0004 | passed |
| problem_n06_01 | 6 | 2 3 0 6 0 0 | 5 0 0 2 6 0 | solved | 734 | 1252 | 0.0087 | passed |
| problem_n06_02 | 6 | 5 0 2 0 6 4 | 3 0 0 6 0 0 | solved | 56 | 78 | 0.0003 | passed |
| problem_n06_03 | 6 | 6 0 2 3 0 0 | 0 0 0 2 1 0 | solved | 849 | 1364 | 0.0044 | passed |
| problem_n06_04 | 6 | 2 0 0 6 3 0 | 0 0 1 0 0 0 | solved | 592 | 933 | 0.0033 | passed |
| problem_n06_05 | 6 | 0 4 1 0 2 3 | 0 0 0 1 6 2 | solved | 1081 | 3147 | 0.0144 | passed |
| problem_n07_01 | 7 | 0 0 7 1 0 3 0 | 3 0 0 7 6 0 0 | solved | 810 | 1038 | 0.0049 | passed |
| problem_n07_02 | 7 | 0 0 6 1 4 0 3 | 5 0 4 6 7 0 0 | solved | 2270 | 3028 | 0.0147 | passed |
| problem_n07_03 | 7 | 0 4 7 5 3 0 0 | 4 7 2 6 0 0 1 | solved | 9527 | 16306 | 0.0793 | passed |
| problem_n07_04 | 7 | 6 4 1 5 0 7 0 | 3 7 0 2 0 1 0 | solved | 5495 | 8095 | 0.0434 | passed |
| problem_n07_05 | 7 | 4 3 1 0 0 7 2 | 0 0 0 5 0 7 1 | solved | 10560 | 18105 | 0.0845 | passed |
| problem_n08_01 | 8 | 8 0 2 7 1 3 0 0 | 6 5 0 8 0 0 1 0 | solved | 122223 | 190342 | 1.5390 | passed |
| problem_n08_02 | 8 | 0 8 1 0 0 5 6 3 | 7 0 0 1 0 4 5 3 | solved | 109768 | 348011 | 4.5847 | passed |
| problem_n08_03 | 8 | 0 8 4 0 3 2 0 0 | 5 0 2 0 8 0 4 6 | solved | 149239 | 293662 | 4.7600 | passed |
| problem_n08_04 | 8 | 3 0 0 2 0 5 1 7 | 0 6 7 0 8 5 2 0 | solved | 102862 | 353051 | 5.5466 | passed |
| problem_n08_05 | 8 | 0 4 2 7 0 0 0 6 | 0 0 0 8 1 5 2 6 | solved | 138857 | 229935 | 3.6675 | passed |
| problem_n09_01 | 9 | 3 0 9 5 1 8 0 7 0 | 7 0 5 9 0 2 8 3 6 | node_limit | â€” | 1000000 | 21.8506 | not_applicable |
| problem_n09_02 | 9 | 7 1 0 6 2 5 0 3 4 | 0 0 2 0 9 7 8 5 0 | solved | 90002 | 105305 | 1.5435 | passed |
| problem_n09_03 | 9 | 3 0 0 9 4 0 8 2 1 | 0 7 0 5 0 0 1 2 6 | node_limit | â€” | 1000000 | 21.0508 | not_applicable |
| problem_n09_04 | 9 | 0 8 4 0 1 7 0 5 0 | 7 0 2 0 4 0 8 9 6 | node_limit | â€” | 1000000 | 20.2781 | not_applicable |
| problem_n09_05 | 9 | 7 0 0 0 0 0 3 6 2 | 0 0 1 8 0 2 9 6 3 | node_limit | â€” | 1000000 | 20.0711 | not_applicable |
| problem_n10_01 | 10 | 0 6 0 10 7 8 0 9 0 1 | 0 9 0 6 7 8 10 0 0 2 | node_limit | â€” | 1000000 | 27.9634 | not_applicable |
| problem_n10_02 | 10 | 2 0 0 1 4 7 0 0 3 0 | 7 4 0 0 2 1 3 5 10 0 | node_limit | â€” | 1000000 | 27.3720 | not_applicable |
| problem_n10_03 | 10 | 3 6 10 0 2 0 9 5 4 0 | 4 0 2 0 9 10 6 3 0 0 | node_limit | â€” | 1000000 | 29.0341 | not_applicable |
| problem_n10_04 | 10 | 7 8 5 10 6 0 3 0 1 0 | 4 5 8 2 6 0 1 0 7 3 | node_limit | â€” | 1000000 | 25.7258 | not_applicable |
| problem_n10_05 | 10 | 0 5 6 3 8 0 2 1 10 4 | 0 7 5 0 6 0 4 2 0 9 | node_limit | â€” | 1000000 | 26.6342 | not_applicable |
| problem_n11_01 | 11 | 0 0 0 11 1 0 2 0 5 7 0 | 0 0 11 0 1 2 9 3 0 4 6 | node_limit | â€” | 1000000 | 32.9521 | not_applicable |
| problem_n11_02 | 11 | 5 9 2 8 0 0 0 10 6 7 3 | 0 9 1 3 6 10 5 11 0 0 0 | node_limit | â€” | 1000000 | 35.5043 | not_applicable |
| problem_n11_03 | 11 | 11 8 2 3 0 5 0 7 0 4 0 | 0 8 11 0 2 5 10 1 0 0 7 | node_limit | â€” | 1000000 | 36.3647 | not_applicable |
| problem_n11_04 | 11 | 0 3 8 0 2 10 0 0 6 11 0 | 10 11 4 9 0 0 0 7 2 5 8 | node_limit | â€” | 1000000 | 30.2469 | not_applicable |
| problem_n11_05 | 11 | 6 0 0 8 7 5 0 0 3 0 10 | 3 4 5 0 0 8 6 0 0 2 7 | node_limit | â€” | 1000000 | 31.8662 | not_applicable |
| problem_n12_01 | 12 | 0 8 12 0 7 0 0 10 3 4 6 0 | 3 4 11 7 1 12 0 0 0 2 6 0 | node_limit | â€” | 1000000 | 41.5982 | not_applicable |
| problem_n12_02 | 12 | 0 12 5 0 6 4 0 10 8 11 7 1 | 0 9 0 3 0 4 2 0 0 0 10 1 | node_limit | â€” | 1000000 | 44.5890 | not_applicable |
| problem_n12_03 | 12 | 8 1 5 3 0 0 0 12 11 2 4 0 | 3 0 9 11 10 12 0 6 0 8 1 0 | node_limit | â€” | 1000000 | 34.6806 | not_applicable |
| problem_n12_04 | 12 | 8 6 4 0 0 9 10 0 5 0 1 0 | 0 0 0 0 0 4 9 2 6 3 1 11 | node_limit | â€” | 1000000 | 38.3652 | not_applicable |
| problem_n12_05 | 12 | 0 4 0 10 6 0 9 0 11 0 3 7 | 4 0 0 9 0 0 8 6 2 5 1 3 | node_limit | â€” | 1000000 | 33.7769 | not_applicable |
| problem_n13_01 | 13 | 0 9 4 12 8 3 1 0 11 0 7 0 5 | 5 0 4 1 8 0 0 9 0 7 0 0 3 | node_limit | â€” | 1000000 | 48.0231 | not_applicable |
| problem_n13_02 | 13 | 0 0 0 12 0 1 0 3 4 8 6 0 9 | 0 0 7 5 1 0 6 13 3 2 4 11 0 | node_limit | â€” | 1000000 | 42.4952 | not_applicable |
| problem_n13_03 | 13 | 0 12 2 1 3 7 4 0 10 13 9 0 8 | 7 13 0 0 6 12 0 0 1 4 8 11 9 | node_limit | â€” | 1000000 | 56.1829 | not_applicable |
| problem_n13_04 | 13 | 0 0 0 5 8 12 1 10 13 7 6 0 0 | 0 12 11 1 0 0 10 9 6 0 7 0 4 | node_limit | â€” | 1000000 | 52.1805 | not_applicable |
| problem_n13_05 | 13 | 10 4 9 5 0 3 0 7 1 0 8 0 0 | 3 1 0 12 13 0 6 11 8 0 0 9 10 | node_limit | â€” | 1000000 | 53.0791 | not_applicable |
| problem_n14_01 | 14 | 10 6 5 7 0 0 0 0 4 3 14 13 0 1 | 6 0 13 2 0 4 5 9 7 0 14 10 0 0 | node_limit | â€” | 1000000 | 63.1358 | not_applicable |
| problem_n14_02 | 14 | 0 0 0 3 10 5 8 9 11 4 0 13 1 0 | 2 0 12 13 4 0 9 1 0 0 6 11 0 0 | node_limit | â€” | 1000000 | 58.8703 | not_applicable |
| problem_n14_03 | 14 | 8 9 0 0 2 1 3 5 10 0 6 7 0 0 | 9 1 0 11 4 3 6 7 10 0 0 2 8 12 | node_limit | â€” | 1000000 | 64.9187 | not_applicable |
| problem_n14_04 | 14 | 7 3 0 1 10 5 12 6 0 14 9 11 8 0 | 4 9 5 0 14 10 13 0 3 1 7 6 0 0 | node_limit | â€” | 1000000 | 60.3466 | not_applicable |
| problem_n14_05 | 14 | 0 10 0 0 2 4 0 7 5 14 3 13 6 0 | 0 3 13 7 8 2 10 9 0 6 0 1 0 4 | node_limit | â€” | 1000000 | 65.8635 | not_applicable |
| problem_n15_01 | 15 | 9 14 13 6 3 11 0 15 10 12 0 0 7 0 0 | 5 15 0 10 9 1 0 3 11 12 0 0 2 13 7 | node_limit | â€” | 1000000 | 80.9338 | not_applicable |
| problem_n15_02 | 15 | 0 7 9 6 14 8 0 0 1 15 13 4 2 3 0 | 13 0 7 1 8 10 0 9 11 4 14 15 0 2 0 | node_limit | â€” | 1000000 | 72.9639 | not_applicable |
| problem_n15_03 | 15 | 5 8 11 1 0 0 6 14 15 7 12 13 2 0 0 | 0 0 15 5 2 8 0 10 7 11 0 1 14 12 13 | node_limit | â€” | 1000000 | 74.7165 | not_applicable |
| problem_n15_04 | 15 | 10 0 12 0 1 8 13 2 6 0 15 4 0 0 14 | 9 0 0 7 0 15 6 12 13 0 10 0 14 5 3 | node_limit | â€” | 1000000 | 78.4660 | not_applicable |
| problem_n15_05 | 15 | 0 0 14 0 6 13 11 9 0 12 0 2 4 10 1 | 0 0 8 9 4 13 14 0 1 7 3 15 11 2 0 | node_limit | â€” | 1000000 | 71.8658 | not_applicable |

</details>

## Wariant rozszerzony â€” DFS kontra BFS

BFS gwarantuje najkrÃ³tszÄ… sekwencjÄ™, natomiast DFS zwraca pierwsze znalezione rozwiÄ…zanie. PorÃ³wnanie wykorzystuje te same 10 problemÃ³w z `N=8`; rzeczywiste rÃ³Å¼nice pamiÄ™ci, czasu i liczby wÄ™zÅ‚Ã³w sÄ… wyliczane poniÅ¼ej.

In [4]:
comparison = [
    row for row in records
    if row['variant'] == 'extended'
    and row['n'] == 8
    and row['algorithm'] in {'dfs', 'bfs'}
]
assert len(comparison) == 20, f'Oczekiwano 20 wynikÃ³w DFS/BFS, otrzymano {len(comparison)}'
comparison_summary = {}
for algorithm in ('dfs', 'bfs'):
    rows = verified([row for row in comparison if row['algorithm'] == algorithm])
    assert len(rows) == 10, f'{algorithm}: nie wszystkie wyniki sÄ… poprawne'
    comparison_summary[algorithm] = {
        'algorithm': algorithm.upper(),
        'solved': len(rows),
        'mean_move_count': round(mean(row['move_count'] for row in rows), 3),
        'mean_expanded_nodes': round(mean(row['expanded_nodes'] for row in rows), 3),
        'mean_elapsed_seconds': round(mean(row['elapsed_seconds'] for row in rows), 6),
    }
show_table(list(comparison_summary.values()), list(next(iter(comparison_summary.values()))))
show_details(
    'PokaÅ¼ wyniki wszystkich 20 uruchomieÅ„ DFS/BFS',
    comparison,
    ['case_id', 'algorithm', 'status', 'move_count', 'expanded_nodes', 'elapsed_seconds', 'verification'],
)
pairs = {}
for row in comparison:
    pairs.setdefault(row['case_id'], {})[row['algorithm']] = row
assert all(pair['bfs']['move_count'] <= pair['dfs']['move_count'] for pair in pairs.values())
strictly_shorter = sum(pair['bfs']['move_count'] < pair['dfs']['move_count'] for pair in pairs.values())
display(Markdown(
    f'BFS znalazÅ‚ krÃ³tszÄ… sekwencjÄ™ w **{strictly_shorter} z 10** przypadkÃ³w. '
    f'Åšrednia dÅ‚ugoÅ›Ä‡ wyniosÅ‚a **{comparison_summary["bfs"]["mean_move_count"]}** dla BFS '
    f'i **{comparison_summary["dfs"]["mean_move_count"]}** dla DFS.'
))

| algorithm | solved | mean_move_count | mean_expanded_nodes | mean_elapsed_seconds |
| --- | --- | --- | --- | --- |
| DFS | 10 | 101573.6000 | 119306.3000 | 2.9124 |
| BFS | 10 | 7.2000 | 52013.6000 | 1.6998 |

<details><summary>PokaÅ¼ wyniki wszystkich 20 uruchomieÅ„ DFS/BFS</summary>

| case_id | algorithm | status | move_count | expanded_nodes | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- |
| problem_n08_01 | dfs | solved | 132673 | 148603 | 2.4064 | passed |
| problem_n08_02 | dfs | solved | 1991 | 1993 | 0.0524 | passed |
| problem_n08_03 | dfs | solved | 38326 | 39901 | 0.7099 | passed |
| problem_n08_04 | dfs | solved | 243938 | 307983 | 7.7731 | passed |
| problem_n08_05 | dfs | solved | 119103 | 131698 | 3.5993 | passed |
| problem_n08_06 | dfs | solved | 219667 | 268137 | 7.1346 | passed |
| problem_n08_07 | dfs | solved | 33378 | 34542 | 0.6628 | passed |
| problem_n08_08 | dfs | solved | 21549 | 21986 | 0.5284 | passed |
| problem_n08_09 | dfs | solved | 20242 | 20610 | 0.5076 | passed |
| problem_n08_10 | dfs | solved | 184869 | 217610 | 5.7496 | passed |
| problem_n08_01 | bfs | solved | 6 | 21715 | 0.8134 | passed |
| problem_n08_02 | bfs | solved | 6 | 2298 | 0.1082 | passed |
| problem_n08_03 | bfs | solved | 6 | 4174 | 0.1647 | passed |
| problem_n08_04 | bfs | solved | 7 | 24735 | 0.8068 | passed |
| problem_n08_05 | bfs | solved | 7 | 27452 | 0.9888 | passed |
| problem_n08_06 | bfs | solved | 9 | 107183 | 3.3506 | passed |
| problem_n08_07 | bfs | solved | 8 | 136890 | 4.2519 | passed |
| problem_n08_08 | bfs | solved | 8 | 61732 | 1.9887 | passed |
| problem_n08_09 | bfs | solved | 8 | 26295 | 0.8893 | passed |
| problem_n08_10 | bfs | solved | 7 | 107662 | 3.6357 | passed |

</details>

BFS znalazÅ‚ krÃ³tszÄ… sekwencjÄ™ w **10 z 10** przypadkÃ³w. Åšrednia dÅ‚ugoÅ›Ä‡ wyniosÅ‚a **7.2** dla BFS i **101573.6** dla DFS.

## IDA* i heurystyki

Pierwsza heurystyka `misplaced` liczy klocki leÅ¼Ä…ce na innym bezpoÅ›rednim podÅ‚oÅ¼u niÅ¼ w celu. Jest dopuszczalna, poniewaÅ¼ pojedynczy ruch zmienia podÅ‚oÅ¼e dokÅ‚adnie jednego klocka, wiÄ™c moÅ¼e usunÄ…Ä‡ najwyÅ¼ej jednÄ… takÄ… niezgodnoÅ›Ä‡.

Druga heurystyka `foundation` dodaje koszt zdjÄ™cia i ponownego uÅ‚oÅ¼enia klockÃ³w, ktÃ³rych podpora jest zgodna z celem, ale sama nie znajduje siÄ™ jeszcze na wÅ‚aÅ›ciwym miejscu. Ruchy te dotyczÄ… innych klockÃ³w niÅ¼ ruchy zliczone przez pierwszÄ… heurystykÄ™, wiÄ™c suma nadal nie przekracza kosztu rozwiÄ…zania. Druga heurystyka dominuje pierwszÄ….

In [5]:
ida = [row for row in records if row['algorithm'] == 'ida_star']
assert len(ida) == 50, f'Oczekiwano 50 wynikÃ³w IDA*, otrzymano {len(ida)}'
expected_heuristics = {'misplaced', 'foundation'}
assert {row['heuristic'] for row in ida} == expected_heuristics
ida_summary = []
for heuristic in sorted(expected_heuristics):
    for n in (6, 8, 10, 12, 15):
        rows = [row for row in ida if row['heuristic'] == heuristic and row['n'] == n]
        assert len(rows) == 5
        ida_summary.append({
            'heuristic': heuristic,
            'n': n,
            'solved': sum(row['status'] == 'solved' for row in rows),
            'node_limit': sum(row['status'] == 'node_limit' for row in rows),
            'mean_expanded_nodes': round(mean(row['expanded_nodes'] for row in rows), 3),
            'mean_elapsed_seconds': round(mean(row['elapsed_seconds'] for row in rows), 6),
        })
show_table(ida_summary, ['heuristic', 'n', 'solved', 'node_limit', 'mean_expanded_nodes', 'mean_elapsed_seconds'])
show_details(
    'PokaÅ¼ peÅ‚ne wyniki IDA* (50 uruchomieÅ„)',
    ida,
    ['case_id', 'n', 'heuristic', 'status', 'move_count', 'expanded_nodes', 'elapsed_seconds', 'verification'],
)

by_case = {}
for row in ida:
    by_case.setdefault(row['case_id'], []).append(row)
comparable = {}
skipped = []
for case_id, rows in by_case.items():
    solved = {row['heuristic']: row for row in verified(rows)}
    if len(expected_heuristics) >= 2 and set(solved) == expected_heuristics:
        comparable[case_id] = solved
    else:
        skipped.append(case_id)
mismatches = {
    case_id: sorted({row['move_count'] for row in rows.values()})
    for case_id, rows in comparable.items()
    if len({row['move_count'] for row in rows.values()}) > 1
}
assert not mismatches, f'Heurystyki zwrÃ³ciÅ‚y rÃ³Å¼ne dÅ‚ugoÅ›ci: {mismatches}'
assert comparable, 'Brak kompletnych par wynikÃ³w IDA* do porÃ³wnania'
print(f'Sprawdzenie zgodnoÅ›ci dÅ‚ugoÅ›ci IDA*: OK ({len(comparable)} porÃ³wnanych par)')
if skipped:
    print(f'PominiÄ™te przypadki (limit lub brak wyniku jednej heurystyki): {len(skipped)}')
improved_better = sum(
    rows['foundation']['expanded_nodes']
    < rows['misplaced']['expanded_nodes']
    for rows in comparable.values()
)
display(Markdown(
    f'WÅ›rÃ³d **{len(comparable)}** kompletnych par silniejsza heurystyka rozwinÄ™Å‚a mniej wÄ™zÅ‚Ã³w '
    f'w **{improved_better}** przypadkach. Dla pozostaÅ‚ych przypadkÃ³w limit jest jawnie pokazany w tabeli.'
))

| heuristic | n | solved | node_limit | mean_expanded_nodes | mean_elapsed_seconds |
| --- | --- | --- | --- | --- | --- |
| foundation | 6 | 5 | 0 | 6 | 0.0004 |
| foundation | 8 | 5 | 0 | 102.2000 | 0.0088 |
| foundation | 10 | 5 | 0 | 243.8000 | 0.0170 |
| foundation | 12 | 5 | 0 | 34194.4000 | 6.6610 |
| foundation | 15 | 4 | 1 | 219480.4000 | 52.0760 |
| misplaced | 6 | 5 | 0 | 8.2000 | 0.0004 |
| misplaced | 8 | 5 | 0 | 2090.8000 | 0.0877 |
| misplaced | 10 | 5 | 0 | 24704 | 1.3076 |
| misplaced | 12 | 3 | 2 | 420780 | 273.4664 |
| misplaced | 15 | 3 | 2 | 436188.8000 | 51.2365 |

<details><summary>PokaÅ¼ peÅ‚ne wyniki IDA* (50 uruchomieÅ„)</summary>

| case_id | n | heuristic | status | move_count | expanded_nodes | elapsed_seconds | verification |
| --- | --- | --- | --- | --- | --- | --- | --- |
| problem_n06_01 | 6 | misplaced | solved | 5 | 13 | 0.0002 | passed |
| problem_n06_02 | 6 | misplaced | solved | 5 | 5 | 0.0004 | passed |
| problem_n06_03 | 6 | misplaced | solved | 5 | 9 | 0.0006 | passed |
| problem_n06_04 | 6 | misplaced | solved | 4 | 4 | 0.0002 | passed |
| problem_n06_05 | 6 | misplaced | solved | 6 | 10 | 0.0004 | passed |
| problem_n08_01 | 8 | misplaced | solved | 7 | 7 | 0.0004 | passed |
| problem_n08_02 | 8 | misplaced | solved | 10 | 8788 | 0.3525 | passed |
| problem_n08_03 | 8 | misplaced | solved | 7 | 19 | 0.0004 | passed |
| problem_n08_04 | 8 | misplaced | solved | 9 | 783 | 0.0395 | passed |
| problem_n08_05 | 8 | misplaced | solved | 8 | 857 | 0.0458 | passed |
| problem_n10_01 | 10 | misplaced | solved | 10 | 93985 | 5.4105 | passed |
| problem_n10_02 | 10 | misplaced | solved | 10 | 774 | 0.0551 | passed |
| problem_n10_03 | 10 | misplaced | solved | 10 | 228 | 0.0116 | passed |
| problem_n10_04 | 10 | misplaced | solved | 11 | 28430 | 1.0557 | passed |
| problem_n10_05 | 10 | misplaced | solved | 9 | 103 | 0.0053 | passed |
| problem_n12_01 | 12 | misplaced | node_limit | â€” | 1000000 | 110.3600 | not_applicable |
| problem_n12_02 | 12 | misplaced | solved | 11 | 87249 | 11.4051 | passed |
| problem_n12_03 | 12 | misplaced | node_limit | â€” | 1000000 | 1243.9687 | not_applicable |
| problem_n12_04 | 12 | misplaced | solved | 11 | 16508 | 1.5827 | passed |
| problem_n12_05 | 12 | misplaced | solved | 11 | 143 | 0.0154 | passed |
| problem_n15_01 | 15 | misplaced | node_limit | â€” | 1000000 | 117.7884 | not_applicable |
| problem_n15_02 | 15 | misplaced | solved | 16 | 3488 | 0.4378 | passed |
| problem_n15_03 | 15 | misplaced | solved | 18 | 78641 | 10.6699 | passed |
| problem_n15_04 | 15 | misplaced | solved | 15 | 98815 | 14.0557 | passed |
| problem_n15_05 | 15 | misplaced | node_limit | â€” | 1000000 | 113.2310 | not_applicable |
| problem_n06_01 | 6 | foundation | solved | 5 | 6 | 0.0005 | passed |
| problem_n06_02 | 6 | foundation | solved | 5 | 5 | 0.0003 | passed |
| problem_n06_03 | 6 | foundation | solved | 5 | 8 | 0.0005 | passed |
| problem_n06_04 | 6 | foundation | solved | 4 | 4 | 0.0003 | passed |
| problem_n06_05 | 6 | foundation | solved | 6 | 7 | 0.0006 | passed |
| problem_n08_01 | 8 | foundation | solved | 7 | 7 | 0.0005 | passed |
| problem_n08_02 | 8 | foundation | solved | 10 | 480 | 0.0421 | passed |
| problem_n08_03 | 8 | foundation | solved | 7 | 7 | 0.0002 | passed |
| problem_n08_04 | 8 | foundation | solved | 9 | 9 | 0.0005 | passed |
| problem_n08_05 | 8 | foundation | solved | 8 | 8 | 0.0007 | passed |
| problem_n10_01 | 10 | foundation | solved | 10 | 15 | 0.0015 | passed |
| problem_n10_02 | 10 | foundation | solved | 10 | 92 | 0.0116 | passed |
| problem_n10_03 | 10 | foundation | solved | 10 | 50 | 0.0034 | passed |
| problem_n10_04 | 10 | foundation | solved | 11 | 1011 | 0.0646 | passed |
| problem_n10_05 | 10 | foundation | solved | 9 | 51 | 0.0040 | passed |
| problem_n12_01 | 12 | foundation | solved | 14 | 134061 | 27.0156 | passed |
| problem_n12_02 | 12 | foundation | solved | 11 | 11 | 0.0007 | passed |
| problem_n12_03 | 12 | foundation | solved | 14 | 36858 | 6.2810 | passed |
| problem_n12_04 | 12 | foundation | solved | 11 | 11 | 0.0019 | passed |
| problem_n12_05 | 12 | foundation | solved | 11 | 31 | 0.0060 | passed |
| problem_n15_01 | 15 | foundation | solved | 15 | 91390 | 20.2531 | passed |
| problem_n15_02 | 15 | foundation | solved | 16 | 1298 | 0.2483 | passed |
| problem_n15_03 | 15 | foundation | solved | 18 | 3501 | 0.8746 | passed |
| problem_n15_04 | 15 | foundation | solved | 15 | 1213 | 0.3618 | passed |
| problem_n15_05 | 15 | foundation | node_limit | â€” | 1000000 | 238.6420 | not_applicable |

</details>

Sprawdzenie zgodnoÅ›ci dÅ‚ugoÅ›ci IDA*: OK (21 porÃ³wnanych par)
PominiÄ™te przypadki (limit lub brak wyniku jednej heurystyki): 4


WÅ›rÃ³d **21** kompletnych par silniejsza heurystyka rozwinÄ™Å‚a mniej wÄ™zÅ‚Ã³w w **18** przypadkach. Dla pozostaÅ‚ych przypadkÃ³w limit jest jawnie pokazany w tabeli.

In [6]:
basic_solved = sum(row['status'] == 'solved' for row in dfs_basic)
ida_limits = sum(row['status'] == 'node_limit' for row in ida)
display(Markdown(
    '## Wnioski\n\n'
    f'DFS rozwiÄ…zaÅ‚ **{basic_solved} z 65** problemÃ³w wariantu podstawowego. Algorytm '
    'uÅ¼ywa kolejnoÅ›ci generatora ruchÃ³w i nie gwarantuje najkrÃ³tszej sekwencji. '
    f'W porÃ³wnaniu dla `N=8` BFS miaÅ‚ Å›redniÄ… dÅ‚ugoÅ›Ä‡ **{comparison_summary["bfs"]["mean_move_count"]}**, '
    f'a DFS **{comparison_summary["dfs"]["mean_move_count"]}**; koszt optymalnoÅ›ci widaÄ‡ w liczbie '
    'rozwiniÄ™tych i przechowywanych stanÃ³w oraz czasie. '
    f'IDA* zakoÅ„czyÅ‚ **{ida_limits} z 50** uruchomieÅ„ limitem. Heurystyka uwzglÄ™dniajÄ…ca blokujÄ…ce '
    'klocki dominuje prostÄ… heurystykÄ™ i czÄ™Å›ciej ogranicza liczbÄ™ rozwiniÄ™Ä‡, zachowujÄ…c tÄ™ samÄ… '
    f'optymalnÄ… dÅ‚ugoÅ›Ä‡ we wszystkich **{len(comparable)}** kompletnych parach.'
))

## Wnioski

DFS rozwiÄ…zaÅ‚ **31 z 65** problemÃ³w wariantu podstawowego. Algorytm uÅ¼ywa kolejnoÅ›ci generatora ruchÃ³w i nie gwarantuje najkrÃ³tszej sekwencji. W porÃ³wnaniu dla `N=8` BFS miaÅ‚ Å›redniÄ… dÅ‚ugoÅ›Ä‡ **7.2**, a DFS **101573.6**; koszt optymalnoÅ›ci widaÄ‡ w liczbie rozwiniÄ™tych i przechowywanych stanÃ³w oraz czasie. IDA* zakoÅ„czyÅ‚ **5 z 50** uruchomieÅ„ limitem. Heurystyka uwzglÄ™dniajÄ…ca blokujÄ…ce klocki dominuje prostÄ… heurystykÄ™ i czÄ™Å›ciej ogranicza liczbÄ™ rozwiniÄ™Ä‡, zachowujÄ…c tÄ™ samÄ… optymalnÄ… dÅ‚ugoÅ›Ä‡ we wszystkich **21** kompletnych parach.